In [ ]:
# ════════════════════════════════════════════════════════
# hERG Inhibition Prediction — Final Solution
# Best PL: 8.8629921078  (top5 Nelder-Mead ensemble)
# Models: descriptor_mlp, desc_fp_maccs_mlp, desc_fp_lgbm,
#         fingerprint_lgbm_fs500, chemberta_mlp_v3
# ════════════════════════════════════════════════════════
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from scipy.optimize import minimize
import numpy.random as npr

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
MIXUP_DIR = '/content/drive/MyDrive/MixUP'
JH_DIR    = f'{MIXUP_DIR}/JH'
TARGET    = 'hERG_inhibition'
DROP_COLS = ['id', 'smiles', TARGET]
N_SPLITS  = 5
TOP_N     = 500

## Model 1: Descriptor MLP
RDKit descriptor (210-dim) → MLP → OOF MAE: 8.902794

In [ ]:
train_d = pd.read_csv(f'{MIXUP_DIR}/train/train_rdkit_descriptor.csv')
test_d  = pd.read_csv(f'{MIXUP_DIR}/test/test_rdkit_descriptor.csv')

feat_d   = [c for c in train_d.columns if c not in DROP_COLS]
X_d      = train_d[feat_d].values.astype(np.float32)
y        = train_d[TARGET].values.astype(np.float32)
X_d_test = test_d[feat_d].values.astype(np.float32)
print(f'Descriptor — Train: {X_d.shape}, Test: {X_d_test.shape}')

In [ ]:
class MLP_Small(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.LayerNorm(64),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

class MLP_Large(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.LayerNorm(1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),       nn.LayerNorm(512),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),        nn.LayerNorm(256),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def get_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mlp_predict(model, X_np):
    model.eval()
    preds = []
    with torch.no_grad():
        for (xb,) in DataLoader(TensorDataset(torch.tensor(X_np)), batch_size=4096):
            preds.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(preds)

def train_mlp(X_raw, y, X_test_raw, ModelClass, lr=3e-4, weight_decay=1e-4,
              dropout=0.2, batch_size=256, epochs=300, patience=30, warmup=10):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_pred   = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_raw))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_raw), 1):
        scaler = StandardScaler()
        X_tr  = scaler.fit_transform(X_raw[tr_idx]).astype(np.float32)
        X_val = scaler.transform(X_raw[val_idx]).astype(np.float32)
        X_te  = scaler.transform(X_test_raw).astype(np.float32)
        y_tr, y_val = y[tr_idx], y[val_idx]
        g = torch.Generator(); g.manual_seed(SEED + fold)
        loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
                            batch_size=batch_size, shuffle=True, generator=g)
        model     = ModelClass(X_tr.shape[1], dropout).to(device)
        criterion = nn.L1Loss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = get_scheduler(optimizer, warmup, epochs)
        best_mae, best_state, wait = float('inf'), None, 0
        for epoch in range(1, epochs + 1):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                criterion(model(xb), yb).backward()
                optimizer.step()
            scheduler.step()
            val_mae = mean_absolute_error(y_val, mlp_predict(model, X_val))
            if val_mae < best_mae:
                best_mae   = val_mae
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    print(f'  early stop @ epoch {epoch}')
                    break
        model.load_state_dict(best_state)
        oof_pred[val_idx] = mlp_predict(model, X_val)
        test_preds += mlp_predict(model, X_te) / N_SPLITS
        print(f'Fold {fold} | best_val_mae={best_mae:.6f}')
    print(f'OOF MAE: {mean_absolute_error(y, oof_pred):.6f}')
    return oof_pred, test_preds

In [ ]:
print('=== Model 1: Descriptor MLP ===')
oof_dm, test_dm = train_mlp(X_d, y, X_d_test, MLP_Small,
                             lr=3e-4, dropout=0.2, patience=30, warmup=10)

pd.DataFrame({'id': train_d['id'], 'hERG_inhibition': y, 'descriptor_mlp_oof': oof_dm}
             ).to_csv(f'{JH_DIR}/oof_descriptor_mlp.csv', index=False)
pd.DataFrame({'id': test_d['id'], 'hERG_inhibition': test_dm}
             ).to_csv(f'{JH_DIR}/submission_descriptor_mlp.csv', index=False)
print('저장 완료')

## Model 2: Desc + FP + MACCS MLP
Descriptor(210) + Fingerprint(2048) + MACCS(166) = 2424-dim → MLP → OOF MAE: 9.192788

In [ ]:
train_fp = pd.read_csv(f'{MIXUP_DIR}/train/train_rdkit_fingerprint.csv')
test_fp  = pd.read_csv(f'{MIXUP_DIR}/test/test_rdkit_fingerprint.csv')

feat_fp  = [c for c in train_fp.columns if c not in DROP_COLS]

def smiles_to_maccs(smiles_list):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        fp  = MACCSkeys.GenMACCSKeys(mol) if mol else [0] * 167
        fps.append(list(fp))
    return np.array(fps, dtype=np.float32)[:, 1:]  # bit 0 제거 → (N, 166)

X_maccs_train = smiles_to_maccs(train_d['smiles'].tolist())
X_maccs_test  = smiles_to_maccs(test_d['smiles'].tolist())

X_dfm      = np.concatenate([X_d, train_fp[feat_fp].values.astype(np.float32), X_maccs_train], axis=1)
X_dfm_test = np.concatenate([X_d_test, test_fp[feat_fp].values.astype(np.float32), X_maccs_test], axis=1)
print(f'Desc+FP+MACCS — Train: {X_dfm.shape}, Test: {X_dfm_test.shape}')

In [ ]:
print('=== Model 2: Desc+FP+MACCS MLP ===')
oof_dfpm, test_dfpm = train_mlp(X_dfm, y, X_dfm_test, MLP_Large,
                                 lr=1e-4, dropout=0.3, patience=50, warmup=20)

pd.DataFrame({'id': train_d['id'], 'hERG_inhibition': y, 'desc_fp_maccs_mlp_oof': oof_dfpm}
             ).to_csv(f'{JH_DIR}/oof_desc_fp_maccs_mlp.csv', index=False)
pd.DataFrame({'id': test_d['id'], 'hERG_inhibition': test_dfpm}
             ).to_csv(f'{JH_DIR}/submission_desc_fp_maccs_mlp.csv', index=False)
print('저장 완료')

## Model 3: Desc + FP LightGBM
Descriptor(210) + Fingerprint(2048) = 2258-dim → LightGBM → OOF MAE: 9.081495

In [ ]:
X_dfp      = np.concatenate([X_d, train_fp[feat_fp].values.astype(np.float32)], axis=1)
X_dfp_test = np.concatenate([X_d_test, test_fp[feat_fp].values.astype(np.float32)], axis=1)
all_cols   = feat_d + feat_fp
X_dfp_df      = pd.DataFrame(X_dfp,      columns=all_cols)
X_dfp_test_df = pd.DataFrame(X_dfp_test, columns=all_cols)
print(f'Desc+FP — Train: {X_dfp_df.shape}, Test: {X_dfp_test_df.shape}')

lgbm_params = dict(
    n_estimators=5000, learning_rate=0.03, max_depth=6, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_dfp_pred   = np.zeros(len(train_d))
test_dfp_preds = np.zeros(len(test_d))

print('=== Model 3: Desc+FP LightGBM ===')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_dfp_df), 1):
    X_tr, X_val = X_dfp_df.iloc[tr_idx], X_dfp_df.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = LGBMRegressor(**lgbm_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(100, verbose=False), log_evaluation(500)])
    oof_dfp_pred[val_idx]  = model.predict(X_val)
    test_dfp_preds        += model.predict(X_dfp_test_df) / N_SPLITS
    print(f'Fold {fold} | best_iter={model.best_iteration_} | MAE={mean_absolute_error(y_val, oof_dfp_pred[val_idx]):.6f}')

print(f'OOF MAE: {mean_absolute_error(y, oof_dfp_pred):.6f}')
pd.DataFrame({'id': train_d['id'], 'hERG_inhibition': y, 'desc_fp_lgbm_oof': oof_dfp_pred}
             ).to_csv(f'{JH_DIR}/oof_desc_fp_lgbm.csv', index=False)
pd.DataFrame({'id': test_d['id'], 'hERG_inhibition': test_dfp_preds}
             ).to_csv(f'{JH_DIR}/submission_desc_fp_lgbm.csv', index=False)
print('저장 완료')

## Model 4: Fingerprint LightGBM FS500
Morgan Fingerprint(2048) → Feature Selection(top 500) → LightGBM → OOF MAE: 9.302554

In [ ]:
X_fp      = train_fp[feat_fp]
X_fp_test = test_fp[feat_fp]
print(f'Fingerprint — Train: {X_fp.shape}, Test: {X_fp_test.shape}')

# Step 1: Feature importance
print('=== Model 4: Fingerprint LightGBM FS500 — Step 1: Feature importance ===')
importance_df = pd.DataFrame(index=feat_fp)
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_fp), 1):
    model = LGBMRegressor(**lgbm_params)
    model.fit(X_fp.iloc[tr_idx], y[tr_idx],
              eval_set=[(X_fp.iloc[val_idx], y[val_idx])],
              callbacks=[early_stopping(100, verbose=False), log_evaluation(500)])
    importance_df[f'fold_{fold}'] = model.feature_importances_
    print(f'Fold {fold} done | best_iter={model.best_iteration_}')

importance_df['mean'] = importance_df.mean(axis=1)
selected_features = importance_df.sort_values('mean', ascending=False).head(TOP_N).index.tolist()
print(f'Selected {len(selected_features)} features')

In [ ]:
# Step 2: Retrain with top-500 features
print('=== Model 4: Fingerprint LightGBM FS500 — Step 2: Retrain ===')
X_fp_sel      = X_fp[selected_features]
X_fp_test_sel = X_fp_test[selected_features]
oof_fp_pred   = np.zeros(len(train_fp))
test_fp_preds = np.zeros(len(test_fp))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_fp_sel), 1):
    X_tr, X_val = X_fp_sel.iloc[tr_idx], X_fp_sel.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = LGBMRegressor(**lgbm_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(100, verbose=False), log_evaluation(500)])
    oof_fp_pred[val_idx]  = model.predict(X_val)
    test_fp_preds        += model.predict(X_fp_test_sel) / N_SPLITS
    print(f'Fold {fold} | best_iter={model.best_iteration_} | MAE={mean_absolute_error(y_val, oof_fp_pred[val_idx]):.6f}')

print(f'OOF MAE: {mean_absolute_error(y, oof_fp_pred):.6f}')
pd.DataFrame({'id': train_fp['id'], 'hERG_inhibition': y, 'fingerprint_lgbm_oof': oof_fp_pred}
             ).to_csv(f'{JH_DIR}/oof_fingerprint_lgbm_fs{TOP_N}.csv', index=False)
pd.DataFrame({'id': test_fp['id'], 'hERG_inhibition': test_fp_preds}
             ).to_csv(f'{JH_DIR}/submission_fingerprint_lgbm_fs{TOP_N}.csv', index=False)
print('저장 완료')

## Model 5: ChemBERTa MLP
ChemBERTa embeddings (768-dim) → MLP → OOF MAE: 9.432980

In [ ]:
train_cb = pd.read_csv(f'{MIXUP_DIR}/train/train_chemberta_feature.csv')
test_cb  = pd.read_csv(f'{MIXUP_DIR}/test/test_chemberta_feature.csv')

feat_cb   = [c for c in train_cb.columns if c not in DROP_COLS]
X_cb      = train_cb[feat_cb].values.astype(np.float32)
X_cb_test = test_cb[feat_cb].values.astype(np.float32)
print(f'ChemBERTa — Train: {X_cb.shape}, Test: {X_cb_test.shape}')

In [ ]:
print('=== Model 5: ChemBERTa MLP ===')
oof_cb, test_cb_pred = train_mlp(X_cb, y, X_cb_test, MLP_Large,
                                  lr=3e-4, dropout=0.2, patience=30, warmup=10)

pd.DataFrame({'id': train_cb['id'], 'hERG_inhibition': y, 'chemberta_mlp_oof': oof_cb}
             ).to_csv(f'{JH_DIR}/oof_chemberta_mlp_v3.csv', index=False)
pd.DataFrame({'id': test_cb['id'], 'hERG_inhibition': test_cb_pred}
             ).to_csv(f'{JH_DIR}/submission_chemberta_mlp_v3.csv', index=False)
print('저장 완료')

## Ensemble: Top5 Multi-start Nelder-Mead
OOF MAE: 8.533049 → Public LB: **8.8629921078**

In [ ]:
sample = pd.read_csv(f'{MIXUP_DIR}/sample_submission.csv')

# OOF 로드
oof_dm_df   = pd.read_csv(f'{JH_DIR}/oof_descriptor_mlp.csv')
oof_dfpm_df = pd.read_csv(f'{JH_DIR}/oof_desc_fp_maccs_mlp.csv')
oof_dfp_df  = pd.read_csv(f'{JH_DIR}/oof_desc_fp_lgbm.csv')
oof_fp_df   = pd.read_csv(f'{JH_DIR}/oof_fingerprint_lgbm_fs{TOP_N}.csv')
oof_cb_df   = pd.read_csv(f'{JH_DIR}/oof_chemberta_mlp_v3.csv')

# test 로드
sub_dm   = pd.read_csv(f'{JH_DIR}/submission_descriptor_mlp.csv')
sub_dfpm = pd.read_csv(f'{JH_DIR}/submission_desc_fp_maccs_mlp.csv')
sub_dfp  = pd.read_csv(f'{JH_DIR}/submission_desc_fp_lgbm.csv')
sub_fp   = pd.read_csv(f'{JH_DIR}/submission_fingerprint_lgbm_fs{TOP_N}.csv')
sub_cb   = pd.read_csv(f'{JH_DIR}/submission_chemberta_mlp_v3.csv')

base = oof_dm_df[['id', 'hERG_inhibition', 'descriptor_mlp_oof']].copy()
base = base.merge(oof_dfpm_df[['id', 'desc_fp_maccs_mlp_oof']], on='id', how='left')
base = base.merge(oof_dfp_df[['id', 'desc_fp_lgbm_oof']],       on='id', how='left')
base = base.merge(oof_fp_df[['id', 'fingerprint_lgbm_oof']],    on='id', how='left')
base = base.merge(oof_cb_df[['id', 'chemberta_mlp_oof']],       on='id', how='left')

test_base = sub_dm[['id', 'hERG_inhibition']].rename(columns={'hERG_inhibition': 'pred_dm'}).copy()
test_base = test_base.merge(sub_dfpm[['id', 'hERG_inhibition']].rename(columns={'hERG_inhibition': 'pred_dfpm'}), on='id', how='left')
test_base = test_base.merge(sub_dfp[['id', 'hERG_inhibition']].rename(columns={'hERG_inhibition': 'pred_dfp'}),   on='id', how='left')
test_base = test_base.merge(sub_fp[['id', 'hERG_inhibition']].rename(columns={'hERG_inhibition': 'pred_fp'}),     on='id', how='left')
test_base = test_base.merge(sub_cb[['id', 'hERG_inhibition']].rename(columns={'hERG_inhibition': 'pred_cb'}),     on='id', how='left')

y_true      = base['hERG_inhibition'].values
model_names = ['descriptor_mlp', 'desc_fp_maccs_mlp', 'desc_fp_lgbm',
               f'fingerprint_fs{TOP_N}', 'chemberta_mlp_v3']

oof_matrix  = np.stack([base['descriptor_mlp_oof'].values,
                         base['desc_fp_maccs_mlp_oof'].values,
                         base['desc_fp_lgbm_oof'].values,
                         base['fingerprint_lgbm_oof'].values,
                         base['chemberta_mlp_oof'].values], axis=1)
test_matrix = np.stack([test_base['pred_dm'].values,
                         test_base['pred_dfpm'].values,
                         test_base['pred_dfp'].values,
                         test_base['pred_fp'].values,
                         test_base['pred_cb'].values], axis=1)

for name, i in zip(model_names, range(5)):
    print(f'{name:<22s} OOF MAE: {mean_absolute_error(y_true, oof_matrix[:, i]):.6f}')

In [ ]:
def normalize_weights(weights):
    w = np.abs(np.array(weights))
    return w / w.sum()

n   = oof_matrix.shape[1]
rng = npr.RandomState(SEED)
maes  = np.array([mean_absolute_error(y_true, oof_matrix[:, i]) for i in range(n)])
w_inv = (1 / maes) / (1 / maes).sum()

x0_list = [np.ones(n) / n, w_inv.copy()] + \
          [rng.dirichlet(np.ones(n)) for _ in range(8)]

best_result = None
for x0 in x0_list:
    res = minimize(
        lambda w: mean_absolute_error(y_true, oof_matrix @ normalize_weights(w)),
        x0, method='Nelder-Mead',
        options={'maxiter': 20000, 'xatol': 1e-6, 'fatol': 1e-6}
    )
    if best_result is None or res.fun < best_result.fun:
        best_result = res

w_opt     = normalize_weights(best_result.x)
oof_opt   = oof_matrix  @ w_opt
test_opt  = test_matrix @ w_opt

print('Optimal weights (multi-start Nelder-Mead)')
for name, w in zip(model_names, w_opt):
    print(f'  {name:<22s} {w:.4f}')
print(f'\nEnsemble OOF MAE: {mean_absolute_error(y_true, oof_opt):.6f}')

sub_final = sample.copy()
sub_final['hERG_inhibition'] = test_opt
sub_final.to_csv(f'{JH_DIR}/submission_final_top5.csv', index=False)
print('\nsubmission_final_top5.csv 저장 완료')
print('Public LB: 8.8629921078')